# Random Forests

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
import matplotlib.pyplot as plt
from sklearn.inspection import PartialDependenceDisplay
from sklearn.metrics import (
    confusion_matrix, 
    ConfusionMatrixDisplay,
    classification_report,
    mean_absolute_error,
    mean_squared_error,
    median_absolute_error,
    explained_variance_score,
    r2_score
)

import numpy as np

# Read dataset
data = pd.read_csv("data/dataset.csv")

# Dataset columns by type for preprocessing
binary_cols = [
    "is_driver",
    "is_public_holiday",
    "is_weekday"
]
gender_cols = ["gender"]
time_col = ["time_of_day"]
severity_col = ["severity"]
numeric = ["age", "temperature_2m", "precipitation", "windspeed_10m", "cloud_cover", "hour", "day", "month", "year"]
one_hot_cols = ["vehicle_type", "accident_type"]

# Preprocessors
gender_encoder = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value="unknown")),
    ("encoder", OrdinalEncoder(categories=[["male", "female", "unknown"]], dtype=int))
])

time_encoder = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value="unknown")),
    ("encoder", OrdinalEncoder(categories=[["morning", "afternoon", "evening", "night", "unknown"]], dtype=int))
])
severity_encoder = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value="unknown")),
    ("encoder", OrdinalEncoder(categories=[["none", "minor", "serious", "fatal", "unknown"]], dtype=int))
])

binary_encoder = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value=-1)),
])

numeric_encoder = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

one_hot_encoder = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value="unknown")),
    ("ohe", OneHotEncoder(handle_unknown="ignore"))
])

# 1. What features predict accident severity?

In [ ]:
x_preprocessor = ColumnTransformer(
    transformers=[
        ("gender", gender_encoder, gender_cols),
        ("time_of_day", time_encoder, time_col),
        ("binary", binary_encoder, binary_cols),
        ("numeric", numeric_encoder, numeric),
        ("one_hot", one_hot_encoder, one_hot_cols)
    ]
)

y_preprocessor = ColumnTransformer(
    transformers=[
        ("severity", severity_encoder, severity_col),
    ]
)

rf = RandomForestClassifier(
    n_estimators=1000,
    max_depth=None,
    class_weight="balanced",
    random_state=42
)

pipeline = Pipeline([
    ("preprocessor", x_preprocessor),
    ("classifier", rf)
])

# Remove unknown severities
severity_data = data[~pd.isna(data["severity"])]

vals = severity_data.drop(columns=["severity"])
target = severity_data[severity_col]

y = severity_encoder.fit_transform(target.values.reshape(-1, 1)).ravel()

x_train, x_test, y_train, y_test = train_test_split(vals, y, test_size=0.2, stratify=y, random_state=42)

In [ ]:
pipeline.fit(x_train, y_train)

y_pred = pipeline.predict(x_test)
print(classification_report(y_test, y_pred, target_names=["none", "minor", "serious", "fatal"]))

In [ ]:
ConfusionMatrixDisplay.from_estimator(
    pipeline,
    x_test,
    y_test,
    display_labels=["none", "minor", "serious", "fatal"],
    cmap="Blues",
)
plt.title("Confusion Matrix")
plt.show()

In [ ]:
feature_names = pipeline.named_steps["preprocessor"].get_feature_names_out()
importances = pipeline.named_steps["classifier"].feature_importances_

importance_df = pd.DataFrame({
    "feature": feature_names,
    "importance": importances
}).sort_values(by="importance", ascending=False)

top_feats = importance_df.head(15)

# Plot top feature importances
plt.figure(figsize=(10, 6))
plt.barh(top_feats["feature"], top_feats["importance"])
plt.gca().invert_yaxis()
plt.xlabel("Feature Importance")
plt.title("Top Feature Importances for Severity")
plt.tight_layout()
plt.show()

In [ ]:
# We use a partial dependence plot to visualize the relationship between top features and the target class "fatal" (3)
PartialDependenceDisplay.from_estimator(
    pipeline, x_train, ["age", "temperature_2m", "day"], target=3
)

# 2. Can we predict accident occurrence from weather variables?

In [ ]:
from datetime import datetime
import requests


url = "https://archive-api.open-meteo.com/v1/archive"

weather_cols = ["temperature_2m", "precipitation", "windspeed_10m", "cloud_cover", "weather_code"]
accident_cols = ["accident_id", "date_time"] + weather_cols
# Aggregate weather data per accident
weather_accident_df = data[accident_cols].groupby("accident_id").aggregate(
    {
        "accident_id": "first",
        "date_time": "first",
        **{col: "mean" for col in weather_cols}
    }
)
weather_accident_df["date"] = pd.to_datetime(weather_accident_df["date_time"]).dt.date

# Get daily accident counts
daily_accidents = (
    weather_accident_df
    .groupby("date")
    .size()
    .reset_index(name="num_accidents")
)

# Binary target
daily_accidents["accident_occurred"] = True

# Getting weather data for missing days from Open-Meteo API
accident_dts = pd.to_datetime(weather_accident_df["date_time"])
start_date = accident_dts.min()
end_date = accident_dts.max()

start_date_str = start_date.strftime("%Y-%m-%d")
end_date_str = end_date.strftime("%Y-%m-%d")

params = {
    "longitude": 14.2135424,
    "latitude": 35.935316,
    "start_date": start_date_str,
    "end_date": end_date_str,
    "hourly": ",".join(weather_cols),
    "timezone": "Europe/Malta"
}
r = requests.get(url, params=params)
weather_data = r.json()

# Create DataFrame from weather data
weather_hourly_df = pd.DataFrame({
    "date_time": pd.to_datetime(weather_data["hourly"]["time"]),
    "temperature_2m": weather_data["hourly"]["temperature_2m"],
    "precipitation": weather_data["hourly"]["precipitation"],
    "windspeed_10m": weather_data["hourly"]["windspeed_10m"],
    "cloud_cover": weather_data["hourly"]["cloud_cover"],
    "weather_code": weather_data["hourly"]["weather_code"],
})

# Aggregate daily weather data
weather_hourly_df["date"] = weather_hourly_df["date_time"].dt.date
daily_weather = (
    weather_hourly_df
    .groupby("date")
    .aggregate({
        "temperature_2m": "mean",
        "precipitation": "mean",
        "windspeed_10m": "mean",
        "cloud_cover": "mean",
        "weather_code": "mean"
    })
    .reset_index()
)

# Add missing days
daily_df = daily_weather.merge(
    daily_accidents[["date", "num_accidents", "accident_occurred"]],
    on="date",
    how="left"
)
# Fill missing values
daily_df["accident_occurred"] = daily_df["accident_occurred"].fillna(False)
daily_df["num_accidents"] = daily_df["num_accidents"].fillna(0).astype(int)

In [ ]:
x_preprocessor = ColumnTransformer(
    transformers=[
        ("weather", numeric_encoder, ["temperature_2m", "precipitation", "windspeed_10m", "cloud_cover"]),
    ]
)

rf = RandomForestClassifier(
    n_estimators=1000,
    class_weight="balanced",
    random_state=42
)

pipeline = Pipeline([
    ("preprocessor", x_preprocessor),
    ("classifier", rf)
])

In [ ]:
from sklearn.metrics import roc_auc_score

vals = daily_df
target = daily_df[["accident_occurred"]]

x_train, x_test, y_train, y_test = train_test_split(vals, target, test_size=0.2, stratify=target, random_state=42)

pipeline.fit(x_train, y_train)

y_prob = pipeline.predict_proba(x_test)[:, 1]
y_pred = pipeline.predict(x_test)

print(classification_report(y_test, y_pred))
print("ROC AUC:", roc_auc_score(y_test, y_prob))


In [ ]:
feature_names = pipeline.named_steps["preprocessor"].get_feature_names_out()
importances = pipeline.named_steps["classifier"].feature_importances_

importance_df = pd.DataFrame({
    "feature": feature_names,
    "importance": importances
}).sort_values(by="importance", ascending=False)

top_feats = importance_df.head(15)

# Plot top feature importances
plt.figure(figsize=(10, 6))
plt.barh(top_feats["feature"], top_feats["importance"])
plt.gca().invert_yaxis()
plt.xlabel("Feature Importance")
plt.title("Top Feature Importances for Accident Occurrence")
plt.show()

In [ ]:
features = [
    "temperature_2m",
    "precipitation",
    "windspeed_10m",
    "cloud_cover"
]

# Plot partial dependence for weather features
PartialDependenceDisplay.from_estimator(
    pipeline,
    x_train,
    features=features,
    grid_resolution=50
)

# 3. Can you predict the severity from personal information?

In [ ]:
driver_info_cols = ["age", "gender", "is_driver"]

x_preprocessor = ColumnTransformer(
    transformers=[
        ("gender", gender_encoder, ["gender"]),
        ("binary", binary_encoder, ["is_driver"]),
        ("numeric", numeric_encoder, ["age"]),
    ]
)

y_preprocessor = ColumnTransformer(
    transformers=[
        ("severity", severity_encoder, severity_col),
    ]
)

rf = RandomForestClassifier(
    n_estimators=1000,
    random_state=42
)

pipeline = Pipeline([
    ("preprocessor", x_preprocessor),
    ("classifier", rf)
])

no_severity = data[~pd.isna(data["severity"])]

vals = no_severity[driver_info_cols]
target = no_severity[severity_col]

# x = x_preprocessor.fit_transform(vals)
y = severity_encoder.fit_transform(target.values.reshape(-1, 1)).ravel()

x_train, x_test, y_train, y_test = train_test_split(vals, y, test_size=0.2, stratify=y, random_state=42)

In [ ]:
pipeline.fit(x_train, y_train)

y_pred = pipeline.predict(x_test)
print(classification_report(y_test, y_pred, target_names=["none", "minor", "serious", "fatal"]))

In [ ]:
ConfusionMatrixDisplay.from_estimator(
    pipeline,
    x_test,
    y_test,
    display_labels=["none", "minor", "serious", "fatal"],
    cmap="Blues",
)
plt.title("Confusion Matrix")
plt.show()

# 4. Can ML models predict periods of high accident frequency using temporal features?

In [ ]:
# Keep only one record per accident
accident_df = (
    data
    .drop_duplicates(subset="accident_id")
    .copy()
)
# Add only date part
accident_df["date"] = pd.to_datetime(accident_df["date_time"]).dt.date
# Add weekend column from weekday column
accident_df["is_weekend"] = ~accident_df["is_weekday"]

# One-hot encode time_of_day
tod_dummies = pd.get_dummies(accident_df["time_of_day"], prefix="tod")
accident_df = pd.concat([accident_df, tod_dummies], axis=1)

# Aggregate daily accidents
daily_accidents = (
    accident_df
    .groupby("date")
    .aggregate({
        "date": "first",
        "is_weekday": "sum",
        "is_weekend": "sum",
        "is_public_holiday": "sum",
        **{col: "sum" for col in tod_dummies.columns}
    })
)

# Add week column (starts on Monday)
daily_accidents["week"] = pd.to_datetime(daily_accidents["date"]).dt.to_period(pd.offsets.Week(weekday=6)).astype(str)

# Aggregate weekly accidents
weekly_accidents = (
    daily_accidents
    .groupby("week")
    .aggregate(
        {
            "date": "first",
            "is_weekday": "sum",
            "is_weekend": "sum",
            "is_public_holiday": "sum",
            **{col: "sum" for col in tod_dummies.columns}
        }
    )
)

# Normalise by number of days
total = weekly_accidents["is_weekday"] + weekly_accidents["is_weekend"]
weekly_accidents["total"] = total
weekly_accidents["is_weekday"] = weekly_accidents["is_weekday"] / 5
weekly_accidents["is_weekend"] = weekly_accidents["is_weekend"] / 2
for col in tod_dummies.columns:
    weekly_accidents[col] = weekly_accidents[col] / total

# Add public holidays info
# Taken from https://www.gov.mt/en/About%20Malta/Pages/Public%20Holidays.aspx
public_holidays = [
    "2023-01-01",
    "2023-02-10",
    "2023-03-19",
    "2023-03-31",
    "2023-04-03",
    "2023-05-01",
    "2023-06-07",
    "2023-06-29",
    "2023-08-15",
    "2023-09-08",
    "2023-09-21",
    "2023-10-08",
    "2023-12-13",
    "2023-12-25",
]
public_holidays = pd.to_datetime(public_holidays).strftime("%m-%d")
calendar_df = pd.DataFrame({
    "date": pd.date_range(start=accident_df["date_time"].min(), end=accident_df["date_time"].max(), freq="D")
}) 
calendar_df["month_day"] = calendar_df["date"].dt.strftime("%m-%d")

calendar_df["is_public_holiday"] = calendar_df["month_day"].isin(public_holidays)
calendar_df["week"] = pd.to_datetime(calendar_df["date"]).dt.to_period(pd.offsets.Week(weekday=6)).astype(str)

# Aggregate weekly calendar info
weekly_calendar_df = (
    calendar_df
    .groupby("week")
    .aggregate(
        {
            "is_public_holiday": "sum"
        }
    )
)

# Normalise public holidays
weekly_accidents["is_public_holiday"] /= weekly_calendar_df["is_public_holiday"].clip(lower=1)
# Insert missing weeks
weekly_accidents = weekly_accidents.reindex(weekly_calendar_df.index)
weekly_accidents = weekly_accidents.fillna(0)
# Set number of public holidays
weekly_accidents["public_holidays"] = weekly_calendar_df["is_public_holiday"]

# Get month from weekly accidents
weekly_accidents["month"] = pd.to_datetime(weekly_accidents["date"]).dt.month

In [ ]:
def regression_report(y_true, y_pred):
    print("Regression Report")
    print("=" * 20)
    print(f"MAE             : {mean_absolute_error(y_true, y_pred):.4f}")
    print(f"Median AE       : {median_absolute_error(y_true, y_pred):.4f}")
    print(f"RMSE            : {np.sqrt(mean_squared_error(y_true, y_pred)):.4f}")
    print(f"Explained Var   : {explained_variance_score(y_true, y_pred):.4f}")
    print(f"R²              : {r2_score(y_true, y_pred):.4f}")

rf = RandomForestRegressor(
    n_estimators=1000,
    max_depth=None,
    random_state=42
)

vals = weekly_accidents[["is_weekday", "is_weekend", "is_public_holiday", "tod_afternoon", "tod_evening", "tod_morning", "tod_night", "public_holidays", "month"]]
target = weekly_accidents[["total"]]

x_train, x_test, y_train, y_test = train_test_split(vals, target, test_size=0.2, random_state=42)

rf.fit(x_train, y_train)

y_pred = rf.predict(x_test)
    
regression_report(y_test, y_pred)
print()
for i in range(len(y_pred)):
    print(f"Predicted: {int(y_pred[i])} ({y_pred[i]:.2f}), Actual: {int(y_test.values[i, 0])}")